# VMD-MFGNN Graph-Collapse Fix — Overnight Experimental Run

**What this notebook is.** A focused, standalone test of a specific fix for
the "Adjacency Collapse" bug diagnosed in `STATUS.md`
("Adjacency Collapse Diagnosed", 2026-08-09): the learned per-band graph
adjacency (`FrequencyGraphConstructor.emb1`/`emb2` in
`src/models/vmd_mfgnn.py`) undergoes isotropic norm collapse under Adam's
coupled L2 weight decay (RMS embedding magnitude fell 32x, from Xavier-init
0.2885 to 0.0089, over 32-43 epochs), driving the softmax adjacency to
uniform (~1/N) regardless of input — i.e. the graph never actually learns
differentiated structure.

**What it is NOT.** It does **not** modify, re-run, or risk the existing,
already-verified main pipeline in any way:
- Code changes (`src/trainer.py`, `src/models/vmd_mfgnn.py`,
  `src/models/pooled_graph_mfgnn.py`) are 100% backward compatible: two new
  config flags (`training.no_decay_graph_embeddings`,
  `model.normalize_graph_embeddings`), both **default `False`** — i.e. off
  by default, so the main pipeline's behavior is byte-for-byte unchanged
  unless a caller explicitly opts in (this notebook is the only caller that
  does).
- Results go to a **completely separate** directory, `results_v2/`
  (never `results/`) — zero risk of overwriting the verified `results/`
  artifacts.
- Google Drive sync targets a **separate** Drive folder,
  `vmd_mfgnn_v2_results_v2` (never the main pipeline's
  `vmd_mfgnn_v2_results`) — zero collision risk.
- The main pipeline's `notebooks/vmd_mfgnn_v2_colab.ipynb` is **not
  modified** by this work at all — see that file if you want to compare.

**The fix under test (both enabled together in this notebook):**
1. `training.no_decay_graph_embeddings=True` — excludes `emb1`/`emb2`
   parameters from Adam's `weight_decay`, removing the constant pull toward
   zero that the diagnosis identified as the proximate mechanical cause.
2. `model.normalize_graph_embeddings=True` — L2-normalizes `emb1`/`emb2` to
   unit norm before the bilinear dot product (`FrequencyGraphConstructor.
   forward`), making the adjacency logits cosine similarities that are, by
   construction, immune to any residual raw-norm drift.

A cheap synthetic (no-GPU) test of both flags together showed the fix holds
`emb1` RMS magnitude essentially flat (ratio to init 1.000 after 40
optimizer epochs on a near-constant-target synthetic task) versus a control
run with both flags off, which shrank to 0.603x of its init RMS over the
same 40 epochs on the same seed/data — i.e. the fix demonstrably resists the
exact isotropic-shrinkage mechanism the diagnosis describes, before spending
any GPU time on it.

**What this notebook trains.** Two models, both with **both** fix flags on,
at the HPO-tuned hyperparameters already found by the main pipeline
(`results/hpo_best_params.json` — read-only reuse, not regenerated here):
- `full_model_graphfix`: `VMDMFGNN`, per-band learned graphs.
- `pooled_graph_graphfix`: `PooledGraphMFGNN`, capacity-matched to
  `full_model_graphfix` via the same `_find_matched_hidden_dim` helper the
  main pipeline's ablation stage uses.

Both train for the same `epochs=200`/`patience=20` early-stopping budget as
the main pipeline (see `configs/default.yaml`) — the real collapse was
observed at epoch 32-43, comfortably inside that budget, so this is a
genuinely comparable test, not a truncated one that couldn't have collapsed
in the first place.

**Expected runtime (best-effort estimate, reasoning from the main
notebook's `RUNBOOK` cell and its one completed real run, which logged
`VMD-MFGNN training done in 1944.9s` ≈ 32 min for a comparable
hidden_dim=128 config):**

| Stage | Estimate | Notes |
|---|---|---|
| Data download + VMD decomposition | 1-3 min (if `data/vmd_modes.npy` cache from a prior run is present/restored from Drive) or 15-45 min (cold) | Identical caching behavior to the main notebook; this notebook does not force a recompute |
| `full_model_graphfix` training | 20-40 min | Same architecture/hidden_dim as the main pipeline's VMD-MFGNN (~32 min observed), so similar order of magnitude |
| `pooled_graph_graphfix` training | 15-35 min | Single pooled graph instead of K per-band graphs — typically somewhat cheaper per epoch than the per-band model, per the main pipeline's ablation timing notes |
| Diagnostic cell | <1 min | Cheap, CPU-bound tensor arithmetic on the trained checkpoint |
| **Total** | **~1-3 hours** (cold-cache worst case pushes toward the top of this range) | Budget for at least one Colab disconnect; both training cells are checkpointed the same way the main pipeline's are |

Do not treat this as a guarantee — it is an estimate reasoned from the one
comparable real run logged in the main notebook, not a fresh measurement.


## Setup 1/5 — Mount Google Drive

Separate Drive folder from the main pipeline (`vmd_mfgnn_v2_results_v2`, not `vmd_mfgnn_v2_results`) — zero collision risk with the main run's synced results.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# NOTE: deliberately a DIFFERENT folder from the main notebook's
# '/content/drive/MyDrive/vmd_mfgnn_v2_results' -- this is the graph-fix
# experiment's own separate Drive space.
DRIVE_ROOT = '/content/drive/MyDrive/vmd_mfgnn_v2_results_v2'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive results root (graphfix, separate from main pipeline):', DRIVE_ROOT)


Mounted at /content/drive
Drive results root (graphfix, separate from main pipeline): /content/drive/MyDrive/vmd_mfgnn_v2_results_v2


## Setup 2/5 — Get the repo

Pick ONE of the two cells below (identical to the main notebook's setup —
the repo-clone/dependency-install process is the same regardless of which
experiment is being run). Either way you end up with the repo at
`/content/copper`.


In [2]:
REPO_URL = 'https://github.com/anmol0705/Copper_Price_Forecasting'  # e.g. 'https://github.com/<you>/copper.git'

# NOTE: this experiment's code lives on the 'graph-fix-experiment' branch,
# NOT 'main' -- main is deliberately kept at the state that produced the
# already-verified results/ and paper/main.tex, with none of this
# experiment's changes. Cloning without -b would pull main and silently
# run the UNFIXED code.
BRANCH = 'graph-fix-experiment'

import subprocess
if not os.path.exists('/content/copper'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, '/content/copper'], check=True)
else:
    print('/content/copper already exists, skipping clone')

In [3]:
# ---- OPTION B: upload a copper.zip (use if you don't have a git remote) ----
# 1. Zip the repo locally: `cd D:\copper && zip -r copper.zip . -x '.git/*'`
# 2. Run this cell, click "Choose Files", and select copper.zip.
# 3. Uncomment the extraction lines.

# from google.colab import files
# uploaded = files.upload()  # select copper.zip
# import zipfile
# with zipfile.ZipFile('copper.zip', 'r') as zf:
#     zf.extractall('/content/copper')
# print('Extracted to /content/copper')


In [4]:
import os
assert os.path.isdir('/content/copper'), (
    "Repo not found at /content/copper -- run Option A or Option B above first "
    "(uncomment the lines in whichever cell matches how you're getting the repo)."
)
os.chdir('/content/copper')
import sys
if '/content/copper' not in sys.path:
    sys.path.insert(0, '/content/copper')
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))


cwd: /content/copper
['.git', '.gitignore', 'COLAB_RUNBOOK.md', 'ORCHESTRATOR_REPORT.md', 'PAPER_RESULTS_PLAN.md', 'RESEARCH_BLUEPRINT.md', 'STATUS.md', 'configs', 'copper_fundamentals.md', 'docs', 'gnn_literature_review.md', 'launch-claude.bat', 'literature_gap_analysis.md', 'literature_review_copper_price_forecasting.md', 'notebooks', 'paper', 'requirements.txt', 'results', 'scripts', 'src', 'vmd-mfgnn-protocol', 'vmd_research.md']


## Setup 3/5 — Install dependencies

In [5]:
# !pip install -q -r requirements.txt

In [6]:
!pip install -q vmdpy optuna yfinance torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 27.5 MB/s eta 0:00:00


## Setup 4/5 — Confirm GPU runtime

In [7]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. In Colab: Runtime > Change runtime type > '
          'Hardware accelerator > GPU (T4 or better recommended). Training will '
          'be much slower on CPU and the runtime estimates above do not apply.')


torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Setup 5/5 — Drive sync helpers

Same helper logic as the main notebook, but pointed at the separate
`DRIVE_ROOT` above and syncing `results_v2/` instead of `results/` —
this notebook never reads or writes anything under `results/` or the main
pipeline's Drive folder.


In [8]:
import shutil
from pathlib import Path

LOCAL_ROOT = Path('/content/copper')
DRIVE_ROOT_P = Path(DRIVE_ROOT)

def sync_to_drive(subpaths=('data', 'results_v2')):
    """Mirror the given local subdirectories into Drive. Cheap/incremental:
    only copies files that don't exist yet or whose size differs."""
    for sub in subpaths:
        src = LOCAL_ROOT / sub
        if not src.exists():
            continue
        dst = DRIVE_ROOT_P / sub
        for root, dirs, filenames in os.walk(src):
            rel = Path(root).relative_to(src)
            (dst / rel).mkdir(parents=True, exist_ok=True)
            for fn in filenames:
                s = Path(root) / fn
                d = dst / rel / fn
                if (not d.exists()) or d.stat().st_size != s.stat().st_size:
                    shutil.copy2(s, d)
    print(f'[sync_to_drive] mirrored {subpaths} -> {DRIVE_ROOT_P}')

def restore_from_drive(subpaths=('data', 'results_v2')):
    """Copy the given subdirectories FROM Drive back into the local repo
    (used at the start of a resumed session)."""
    for sub in subpaths:
        src = DRIVE_ROOT_P / sub
        if not src.exists():
            continue
        dst = LOCAL_ROOT / sub
        for root, dirs, filenames in os.walk(src):
            rel = Path(root).relative_to(src)
            (dst / rel).mkdir(parents=True, exist_ok=True)
            for fn in filenames:
                s = Path(root) / fn
                d = dst / rel / fn
                if (not d.exists()) or d.stat().st_size != s.stat().st_size:
                    shutil.copy2(s, d)
    print(f'[restore_from_drive] restored {subpaths} <- {DRIVE_ROOT_P}')

# Pull back anything from a previous (possibly interrupted) session of THIS
# experiment before doing any work below. Note: 'data' is the shared VMD
# cache (harmless/read-mostly to reuse across experiments); 'results_v2'
# is this experiment's own private output tree.
restore_from_drive()


[restore_from_drive] restored ('data', 'results_v2') <- /content/drive/MyDrive/vmd_mfgnn_v2_results_v2


## STEP 1: Data (reuse the main pipeline's VMD cache if present)

Identical data pipeline to the main notebook: full 2010-01-01..2025-12-31
range, real Yahoo Finance downloads, leakage-safe expanding-window VMD
(`debug_fast=False`). `build_vmd_modes()` has its own on-disk cache
(`data/vmd_modes.npy` + `data/vmd_modes_meta.json`) keyed by a hash of the
input data/params — if you already ran the main notebook (or a prior run of
this one) and that cache is present locally or on Drive, this cell loads it
instantly instead of recomputing the slow VMD decomposition. This notebook
only *reads* `data/`; it does not write anything under `results/`.


In [9]:
import src.data_pipeline as dp

# Drop the confirmed-dead zinc/nickel tickers (same fix as the main notebook)
dp.TICKERS = {k: v for k, v in dp.TICKERS.items() if k not in ("zinc", "nickel")}
dp.VARIABLE_NAMES = list(dp.TICKERS.keys())

import pandas as pd
_orig_download = dp.DataDownloader.download
def _patched_download(self):
    import yfinance as yf
    frames = {}
    for name, ticker in dp.TICKERS.items():
        try:
            data = yf.download(ticker, start=self.start, end=self.end, progress=False, auto_adjust=True)
            close = data["Close"]
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            close = close.dropna()
            if len(close) > 0:
                frames[name] = close
                print(f"  {name} ({ticker}): {len(close)} rows")
            else:
                print(f"  WARNING: {name} ({ticker}): no data")
        except Exception as e:
            print(f"  WARNING: {name} ({ticker}): failed - {e}")
    if len(frames) < len(dp.TICKERS):
        missing = set(dp.TICKERS) - set(frames)
        raise ValueError(f"Missing data for: {missing} -- fix before proceeding")
    df = pd.DataFrame(frames).ffill(limit=5).dropna()
    df.index.name = "date"
    self.cache_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(self.cache_path)
    print(f"Saved {len(df)} rows")
    return df
dp.DataDownloader.download = _patched_download


In [10]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)

from src.utils import load_config, set_seed
from src.data_pipeline import create_datasets

config = load_config('configs/default.yaml')
set_seed(config['training']['seed'])

print('Date range:', config['data']['start_date'], '->', config['data']['end_date'])
print('Tickers:', config['data']['tickers'])
print('VMD K:', config['vmd']['K'], '| refit_interval:', config['vmd']['refit_interval'])

data = create_datasets(config, debug_fast=False)
print('Variables:', data['variable_names'])
print('Train/Val/Test samples:', len(data['train_ds']), len(data['val_ds']), len(data['test_ds']))


Date range: 2010-01-01 -> 2025-12-31
Tickers: {'copper': 'HG=F', 'aluminum': 'ALI=F', 'gold': 'GC=F', 'oil': 'CL=F', 'dxy': 'DX-Y.NYB', 'sp500': '^GSPC', 'vix': '^VIX', 'us10y': '^TNX'}
VMD K: 5 | refit_interval: 21
  copper (HG=F): 4023 rows
  aluminum (ALI=F): 2895 rows
  gold (GC=F): 4022 rows
  oil (CL=F): 4023 rows
  dxy (DX-Y.NYB): 4024 rows
  sp500 (^GSPC): 4023 rows
  vix (^VIX): 4023 rows
  us10y (^TNX): 4021 rows
Saved 2914 rows
Variables: ['copper', 'aluminum', 'gold', 'oil', 'dxy', 'sp500', 'vix', 'us10y']
Train/Val/Test samples: 1343 505 984


## STEP 2: Load tuned hyperparameters (`results/hpo_best_params.json`)

Read-only reuse of the main pipeline's already-completed HPO search results
(this notebook does not run HPO itself and does not write anything under
`results/`). If this file doesn't exist yet, run the main notebook's HPO
stage first (or fall back to `configs/default.yaml`'s plain `model:`
settings, which the `except` branch below does automatically).


In [11]:
import json
import copy
from pathlib import Path

hpo_path = Path('results/hpo_best_params.json')
if hpo_path.exists():
    with open(hpo_path) as f:
        best_params = json.load(f)
    print('Loaded tuned hyperparameters from', hpo_path, ':', best_params)
else:
    print(f'WARNING: {hpo_path} not found -- falling back to configs/default.yaml '
          f'model: settings unchanged. Run the main notebook\'s HPO stage first '
          f'for a genuinely comparable (tuned) test.')
    mc0 = config['model']
    best_params = {
        'hidden_dim': mc0['hidden_dim'], 'num_heads': mc0['num_heads'],
        'dropout': mc0['dropout'], 'num_gnn_layers': mc0['num_gnn_layers'],
        'learning_rate': config['training']['learning_rate'],
    }

# Build the graphfix config: same tuned model/training settings as the main
# pipeline would use, PLUS both new fix flags turned ON. epochs/patience are
# left at configs/default.yaml's values (200/20) -- the SAME early-stopping
# budget as the main pipeline, so this is a genuinely comparable test at the
# epoch counts (32-43) where the real collapse was observed, not a truncated
# run that couldn't have collapsed in the first place.
mc = dict(config['model'])
mc.update({k: best_params[k] for k in ['hidden_dim', 'num_heads', 'dropout', 'num_gnn_layers']})
mc['normalize_graph_embeddings'] = True  # fix #2

config_graphfix = copy.deepcopy(config)
config_graphfix['model'] = mc
config_graphfix['training']['learning_rate'] = best_params['learning_rate']
config_graphfix['training']['no_decay_graph_embeddings'] = True  # fix #1

print('graphfix model config:', mc)
print('graphfix training config:', config_graphfix['training'])


Loaded tuned hyperparameters from results/hpo_best_params.json : {'hidden_dim': 128, 'num_heads': 2, 'learning_rate': 0.0015304852121831463, 'dropout': 0.061612603179999434, 'num_gnn_layers': 1}
graphfix model config: {'hidden_dim': 128, 'num_heads': 2, 'num_gnn_layers': 1, 'temporal_module': 'lstm', 'temporal_layers': 2, 'dropout': 0.061612603179999434, 'graph_type': 'learned', 'normalize_graph_embeddings': True}
graphfix training config: {'batch_size': 32, 'epochs': 200, 'learning_rate': 0.0015304852121831463, 'weight_decay': 1e-05, 'patience': 20, 'scheduler': 'cosine', 'seed': 42, 'no_decay_graph_embeddings': True}


## STEP 3: Train `full_model_graphfix` (VMDMFGNN, per-band graphs, both fixes ON)

Checkpointed via `VMDMFGNNTrainer.fit(..., checkpoint_path=...)` under
`results_v2/checkpoints/` (trainer-native resume, same mechanism the
main pipeline uses under `results/checkpoints/` -- completely separate
files, so there is no interaction between the two).


In [12]:
import time
import numpy as np
import torch

from src.utils import get_device, set_seed
from src.trainer import VMDMFGNNTrainer
from src.models.vmd_mfgnn import VMDMFGNN

set_seed(config['training']['seed'])
device = str(get_device())
horizons = data['horizons']
num_vars = data['num_vars']
num_modes = data['num_modes']

ckpt_dir = Path('results_v2/checkpoints')
pred_dir = Path('results_v2/predictions')
interp_dir = Path('results_v2/interpretability')
for d in (ckpt_dir, pred_dir, interp_dir):
    d.mkdir(parents=True, exist_ok=True)

full_model_graphfix = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'],
    temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons,
    graph_type='learned',
    normalize_graph_embeddings=True,  # fix #2
)
full_params_graphfix = sum(p.numel() for p in full_model_graphfix.parameters() if p.requires_grad)
print(f'full_model_graphfix param count: {full_params_graphfix:,} (hidden_dim={mc["hidden_dim"]})')

trainer_full = VMDMFGNNTrainer(full_model_graphfix, config_graphfix)  # fix #1 via config_graphfix['training']

t0 = time.time()
history_full = trainer_full.fit(
    data['train_loader'], data['val_loader'],
    checkpoint_path=ckpt_dir / 'full_model_graphfix.pt')
if history_full.get('resumed_from_checkpoint'):
    print(f"[resume] loaded checkpoint -- training skipped")
else:
    print(f'full_model_graphfix training done in {time.time() - t0:.1f}s, '
          f'{len(history_full["train_loss"])} epochs run')

test_results_full = trainer_full.evaluate(data['test_loader'])
test_results_full['name'] = 'full_model_graphfix'
print('full_model_graphfix test results:', test_results_full)

# predict() also (re)populates attention/graph state for get_learned_graphs() below
full_preds = trainer_full.predict(data['test_loader'])
test_ys = [y.numpy() if isinstance(y, torch.Tensor) else y for _, y in data['test_loader']]
test_true = np.concatenate(test_ys, axis=0)
for i, h in enumerate(horizons):
    np.save(pred_dir / f'full_model_graphfix_{h}.npy', full_preds[str(h)])
    np.save(pred_dir / f'full_model_graphfix_{h}_true.npy', test_true[:, i])

learned_graphs_full = full_model_graphfix.get_learned_graphs()
torch.save(learned_graphs_full, interp_dir / 'full_model_graphfix_learned_graphs.pt')

from src.utils import save_results
save_results(test_results_full, pred_dir / 'full_model_graphfix_metrics.json')

sync_to_drive(('data', 'results_v2'))


full_model_graphfix param count: 1,461,124 (hidden_dim=128)
full_model_graphfix training done in 1242.2s, 21 epochs run
full_model_graphfix test results: {'h1': {'rmse': np.float64(0.018811336294606868), 'mae': 0.013068287633359432, 'mape': np.float32(178.65308), 'r2': -0.10973560810089111, 'da': np.float64(47.357723577235774)}, 'h5': {'rmse': np.float64(0.03980283863211772), 'mae': 0.028157545253634453, 'mape': np.float32(131.0599), 'r2': 0.0040915608406066895, 'da': np.float64(51.72764227642277)}, 'h10': {'rmse': np.float64(0.057092471608756086), 'mae': 0.04283545911312103, 'mape': np.float32(211.2754), 'r2': -0.07137858867645264, 'da': np.float64(44.3089430894309)}, 'h22': {'rmse': np.float64(0.07617269417737975), 'mae': 0.05672653391957283, 'mape': np.float32(157.59033), 'r2': -0.010851502418518066, 'da': np.float64(54.67479674796748)}, 'avg_mse': np.float64(0.00274999049725011), 'name': 'full_model_graphfix'}


TypeError: Object of type float32 is not JSON serializable

In [13]:
from src.utils import save_results
save_results(test_results_full, pred_dir / 'full_model_graphfix_metrics.json')
sync_to_drive(('data', 'results_v2'))
print('Saved metrics and synced.')

[sync_to_drive] mirrored ('data', 'results_v2') -> /content/drive/MyDrive/vmd_mfgnn_v2_results_v2
Saved metrics and synced.


## STEP 4: Train `pooled_graph_graphfix` (PooledGraphMFGNN, both fixes ON, capacity-matched)

Uses `src.trainer._find_matched_hidden_dim` (the same helper the main
pipeline's ablation stage uses) to pick a `hidden_dim` for `PooledGraphMFGNN`
whose parameter count is close to `full_model_graphfix`'s -- so any
adjacency-collapse difference between the two is attributable to the
per-band-vs-pooled graph design, not incidental model size.


In [14]:
from src.models.pooled_graph_mfgnn import PooledGraphMFGNN, count_parameters
from src.trainer import _find_matched_hidden_dim

def build_pooled_graphfix(h):
    return PooledGraphMFGNN(
        num_vars=num_vars, num_modes=num_modes, hidden_dim=h,
        num_heads=mc['num_heads'], num_gnn_layers=mc['num_gnn_layers'],
        temporal_layers=mc['temporal_layers'], dropout=mc['dropout'],
        horizons=horizons, graph_type='learned',
        normalize_graph_embeddings=True,  # fix #2
    )

matched_h, matched_c = _find_matched_hidden_dim(
    build_pooled_graphfix, full_params_graphfix, mc['num_heads'])
print(f'matched pooled_graph_graphfix to hidden_dim={matched_h}, {matched_c:,} params '
      f'vs full_model_graphfix\'s {full_params_graphfix:,} '
      f'({(matched_c - full_params_graphfix) / full_params_graphfix * 100:+.1f}%)')

set_seed(config['training']['seed'])
pooled_graph_graphfix = build_pooled_graphfix(matched_h)

trainer_pooled = VMDMFGNNTrainer(pooled_graph_graphfix, config_graphfix)  # fix #1

t0 = time.time()
history_pooled = trainer_pooled.fit(
    data['train_loader'], data['val_loader'],
    checkpoint_path=ckpt_dir / 'pooled_graph_graphfix.pt')
if history_pooled.get('resumed_from_checkpoint'):
    print(f"[resume] loaded checkpoint -- training skipped")
else:
    print(f'pooled_graph_graphfix training done in {time.time() - t0:.1f}s, '
          f'{len(history_pooled["train_loss"])} epochs run')

test_results_pooled = trainer_pooled.evaluate(data['test_loader'])
test_results_pooled['name'] = 'pooled_graph_graphfix'
print('pooled_graph_graphfix test results:', test_results_pooled)

pooled_preds = trainer_pooled.predict(data['test_loader'])
for i, h in enumerate(horizons):
    np.save(pred_dir / f'pooled_graph_graphfix_{h}.npy', pooled_preds[str(h)])
    np.save(pred_dir / f'pooled_graph_graphfix_{h}_true.npy', test_true[:, i])

learned_graph_pooled = pooled_graph_graphfix.get_learned_graph()
if learned_graph_pooled is not None:
    torch.save(learned_graph_pooled, interp_dir / 'pooled_graph_graphfix_learned_graph.pt')

from src.utils import save_results
save_results(test_results_pooled, pred_dir / 'pooled_graph_graphfix_metrics.json')

sync_to_drive(('data', 'results_v2'))


matched pooled_graph_graphfix to hidden_dim=276, 1,456,166 params vs full_model_graphfix's 1,461,124 (-0.3%)
pooled_graph_graphfix training done in 957.3s, 71 epochs run
pooled_graph_graphfix test results: {'h1': {'rmse': np.float64(0.017867016620354694), 'mae': 0.011971891857683659, 'mape': np.float32(101.75856), 'r2': -0.0011156797409057617, 'da': np.float64(48.3739837398374)}, 'h5': {'rmse': np.float64(0.0402666187827094), 'mae': 0.0283505842089653, 'mape': np.float32(126.48879), 'r2': -0.01925206184387207, 'da': np.float64(46.95121951219512)}, 'h10': {'rmse': np.float64(0.05540498456596414), 'mae': 0.040146347135305405, 'mape': np.float32(117.853455), 'r2': -0.008980870246887207, 'da': np.float64(44.207317073170735)}, 'h22': {'rmse': np.float64(0.07662007018783257), 'mae': 0.0581456795334816, 'mape': np.float32(132.00665), 'r2': -0.02276003360748291, 'da': np.float64(44.71544715447154)}, 'avg_mse': np.float64(0.0027202445853617974), 'name': 'pooled_graph_graphfix'}
[sync_to_drive] 

In [15]:
from src.utils import save_results
save_results(test_results_pooled, pred_dir / 'pooled_graph_graphfix_metrics.json')
sync_to_drive(('data', 'results_v2'))
print('Saved metrics and synced.')

[sync_to_drive] mirrored ('data', 'results_v2') -> /content/drive/MyDrive/vmd_mfgnn_v2_results_v2
Saved metrics and synced.


## STEP 5: Diagnostic — did the fix actually resolve the collapse?

Reloads the just-trained `full_model_graphfix` checkpoint from disk (not
just the in-memory object above, so this cell is also a standalone sanity
check that the checkpoint round-trips correctly) and computes the same
quantities the original diagnostic subagent used on the unfixed checkpoints
(see `STATUS.md` "Adjacency Collapse Diagnosed"):

1. **RMS magnitude of `emb1`/`emb2`** per band, compared against the
   Xavier-init reference (~0.2885) the diagnosis measured. A collapsed model
   would show something near the diagnosed 0.0089 (a ~32x shrinkage); a
   fixed model should stay much closer to its init scale.
2. **Softmax spread** of the resulting per-band adjacency (std of the
   full (pre-topk) softmax row -- a collapsed/uniform adjacency has std
   -> 0; genuine structure has non-trivial std).
3. **Fraction of kept (post-topk) edges within a tight tolerance of exactly
   1/N** -- the diagnosis's smoking-gun signature of collapse (topk
   sparsification selecting on 5th-decimal-place noise while every entry is
   functionally uniform). A low fraction here means edges are genuinely
   differentiated, not just noise-broken ties.

Prints a clear PASS/FAIL-style verdict per band and overall.


In [16]:
import torch
import torch.nn.functional as F
from src.models.vmd_mfgnn import VMDMFGNN

XAVIER_INIT_RMS_REFERENCE = 0.2885  # from STATUS.md's diagnosis (measured at init)
COLLAPSED_RMS_REFERENCE = 0.0089    # from STATUS.md's diagnosis (measured post-collapse)
UNIFORM_TOL = 1e-3                  # how close to 1/N counts as "collapsed to uniform"

diag_model = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=mc['hidden_dim'], num_heads=mc['num_heads'],
    num_gnn_layers=mc['num_gnn_layers'], temporal_layers=mc['temporal_layers'],
    dropout=mc['dropout'], horizons=horizons, graph_type='learned',
    normalize_graph_embeddings=True,
)
ckpt = torch.load(ckpt_dir / 'full_model_graphfix.pt', map_location='cpu')
state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
diag_model.load_state_dict(state)
diag_model.eval()

N = num_vars
print(f'{"band":>5} | {"emb1_rms":>10} | {"emb2_rms":>10} | {"rms/init":>9} | '
      f'{"softmax_std":>12} | {"frac_near_1/N":>14} | verdict')
print('-' * 90)

band_verdicts = []
for k, gc in enumerate(diag_model.graph_constructors):
    e1 = gc.emb1.weight.detach()
    e2 = gc.emb2.weight.detach()
    emb1_rms = e1.pow(2).mean().sqrt().item()
    emb2_rms = e2.pow(2).mean().sqrt().item()
    rms_avg = (emb1_rms + emb2_rms) / 2
    rms_ratio = rms_avg / XAVIER_INIT_RMS_REFERENCE

    # Full (pre-topk) softmax adjacency, matching the diagnosis's own metric.
    e1n, e2n = F.normalize(e1, dim=-1), F.normalize(e2, dim=-1)  # gc.normalize_embeddings is True here
    full_adj = torch.softmax(F.relu(e1n @ e2n.T), dim=-1)
    softmax_std = full_adj.std().item()

    # Post-topk kept edges: what fraction sit within UNIFORM_TOL of exactly 1/N?
    kept_adj = gc.get_adjacency()
    nonzero = kept_adj[kept_adj > 0]
    frac_near_uniform = (nonzero.sub(1.0 / N).abs() < UNIFORM_TOL).float().mean().item() if len(nonzero) else float('nan')

    collapsed = (rms_ratio < 0.5) or (frac_near_uniform > 0.8)
    verdict = 'COLLAPSED' if collapsed else 'OK (differentiated)'
    band_verdicts.append(not collapsed)
    print(f'{k:>5} | {emb1_rms:>10.5f} | {emb2_rms:>10.5f} | {rms_ratio:>9.3f} | '
          f'{softmax_std:>12.6f} | {frac_near_uniform:>14.3f} | {verdict}')

print('-' * 90)
n_ok = sum(band_verdicts)
print(f'\n{n_ok}/{num_modes} bands show a genuinely differentiated (non-collapsed) adjacency.')
print(f'Reference: Xavier-init RMS ~ {XAVIER_INIT_RMS_REFERENCE}, '
      f'diagnosed-collapse RMS ~ {COLLAPSED_RMS_REFERENCE} '
      f'({COLLAPSED_RMS_REFERENCE / XAVIER_INIT_RMS_REFERENCE:.3f}x init).')

if n_ok == num_modes:
    print('\n=== VERDICT: FIX APPEARS TO HAVE WORKED -- no band shows the diagnosed '
          'collapse-to-uniform pattern. ===')
elif n_ok == 0:
    print('\n=== VERDICT: FIX DID NOT RESOLVE THE COLLAPSE -- all bands still show the '
          'diagnosed pattern. The conceptual explanation in STATUS.md (uniform '
          'adjacency is already near-optimal at N={} nodes without an explicit '
          'sparsity/entropy regularizer) may be the dominant effect. ==='.format(N))
else:
    print(f'\n=== VERDICT: PARTIAL -- {n_ok}/{num_modes} bands avoided collapse. Mixed '
          'result; inspect per-band table above before drawing conclusions. ===')


 band |   emb1_rms |   emb2_rms |  rms/init |  softmax_std |  frac_near_1/N | verdict
------------------------------------------------------------------------------------------
    0 |    0.30253 |    0.27461 |     1.000 |     0.017241 |          0.188 | OK (differentiated)
    1 |    0.27951 |    0.27467 |     0.960 |     0.017942 |          0.000 | OK (differentiated)
    2 |    0.29225 |    0.27336 |     0.980 |     0.012986 |          0.094 | OK (differentiated)
    3 |    0.29267 |    0.26984 |     0.975 |     0.015625 |          0.062 | OK (differentiated)
    4 |    0.28097 |    0.28957 |     0.989 |     0.020019 |          0.000 | OK (differentiated)
------------------------------------------------------------------------------------------

5/5 bands show a genuinely differentiated (non-collapsed) adjacency.
Reference: Xavier-init RMS ~ 0.2885, diagnosed-collapse RMS ~ 0.0089 (0.031x init).

=== VERDICT: FIX APPEARS TO HAVE WORKED -- no band shows the diagnosed collapse-to-unif

## Final sync + download

In [17]:
sync_to_drive(('data', 'results_v2'))
print('Final sync complete. Drive folder:', DRIVE_ROOT)


[sync_to_drive] mirrored ('data', 'results_v2') -> /content/drive/MyDrive/vmd_mfgnn_v2_results_v2
Final sync complete. Drive folder: /content/drive/MyDrive/vmd_mfgnn_v2_results_v2


In [18]:
# Optional: zip results_v2/ and download directly (in addition to the Drive copy).
import shutil
from google.colab import files
shutil.make_archive('results_v2', 'zip', 'results_v2')
files.download('results_v2.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Done

Check, in order:
1. The STEP 5 diagnostic cell's printed verdict (FIX APPEARS TO HAVE WORKED /
   DID NOT RESOLVE / PARTIAL) -- this is the headline result.
2. `results_v2/predictions/full_model_graphfix_metrics.json` and
   `pooled_graph_graphfix_metrics.json` -- test-set RMSE/MAE/DA per horizon,
   for comparison against the main pipeline's unfixed `results/all_results.json`
   `VMD-MFGNN` entry (same test set, same horizons).
3. `results_v2/interpretability/full_model_graphfix_learned_graphs.pt`
   and `pooled_graph_graphfix_learned_graph.pt` -- the raw learned adjacency
   matrices, for any further manual inspection beyond the diagnostic cell's
   summary stats.
4. This notebook and its `results_v2/` output are entirely separate
   from the main pipeline's `notebooks/vmd_mfgnn_v2_colab.ipynb` and
   `results/` -- nothing here needs to be reconciled with or merged into the
   main, already-verified results.
